# LLM techniques — structured outputs, function calling, fine-tuning, and a signpost to RAG

Three ideas from the slides, turned into code you can run — making a model return data instead of prose, letting it look numbers up through your own Python, and adapting a small open-weights model with a genuine LoRA training run — and a fourth, retrieval, whose place this notebook marks and whose code is in Case Study 2.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simonhatzesberger/advanced-genai-actuarial/blob/main/notebooks/02_llm_techniques/02_llm_techniques.ipynb)

EAA seminar **E0572 — Advanced Applications of Generative AI in Actuarial Science**, Vilnius, 1–2 October 2026. Lecturer: Dr Simon Hatzesberger.

Repository: [simonhatzesberger/advanced-genai-actuarial](https://github.com/simonhatzesberger/advanced-genai-actuarial)

This is the second notebook of the series: `01_llm_basics` has already covered the first API call, tokens and cost, prompt engineering and where LLMs fall down, so none of that is repeated here. Section 2 reads ten synthetic car-insurance contracts from the RISCBAC corpus, and section 3 lets the model query a small database of German mortality tables. Both data sets are in `data/` next to the notebook; `data/README.md` gives their sources.

**Reading it and running it.** Like every notebook in this series, it ships with its outputs. Sections 2 and 3 call the model: 19 requests, which cost USD 0.0116 in the saved run of 23 September 2026 at list prices, or USD 0.0148 had none of their input been cached. Without a key, or when a call fails, they replay the replies of that run from `data/recordings_02.json` and say so in an `[ATTENTION]` line; a prompt you change has no recording and needs a key. Section 4 sends nothing: it trains a model on the machine the notebook runs on, and the first time you reach it, it downloads about 1 GB. Its saved outputs are those of the run of 22 September 2026. With a key, a rerun of sections 2 and 3 can give different answers, and the commentary describes the saved run.

## Learning objectives

After working through this notebook you will be able to:

- **force** a model's answer into a schema you define — dates, numbers, yes/no flags, fixed categories, an ordered level, optional fields and lists of nested objects — and read ten contracts straight into a `pandas` table;
- **check** what comes back with code: premiums that must add up, a tax rate, a driver's age, an address that does not fit the policy form;
- **let** a model query a database through two functions you wrote, keep the database read-only and the SQL parameterised, and check every number the model hands back;
- **fine-tune** a small open-weights model with LoRA, and say how many parameters that trains;
- **judge** honestly what twenty training examples do and do not buy you;
- **decide** which of the four techniques fixes the problem in front of you.

## Contents

1. [Setup](#1-setup)
2. [Structured outputs, in three stages](#2-structured-outputs-in-three-stages)
3. [Function calling on a mortality database](#3-function-calling-on-a-mortality-database)
4. [Fine-tuning with LoRA: a real run](#4-fine-tuning-with-lora-a-real-run)
5. [Retrieval-augmented generation: a signpost](#5-retrieval-augmented-generation-a-signpost)
6. [When to use which](#6-when-to-use-which)

---

## 1 Setup

One cell, run once. It builds on the setup cell of `01_llm_basics` — the same pinned model, the same list prices, the same `find_api_key()` from `00_getting_started` and the same `ask()` — so that from here on you read *what we asked* rather than the plumbing around it. Four things are new:

- **`parse()`**, the structured-output twin of `ask()`: it sends a Pydantic class to `client.responses.parse` as `text_format` and returns the validated Python object;
- **a ledger** that turns the token counts of every reply into money at the list prices of 15 September 2026, including the lower price of input that the provider has cached from an earlier request;
- **`data_file()`**, which finds the notebook's data files in `data/` next to it, or downloads them once from GitHub, and checks each against its published checksum;
- **`respond()`**, through which every request goes. With a key it calls the model. Without one, or when a call fails, it replays the reply that the saved run got to the same request — identical in model, instructions, input, tools and schema — from `data/recordings_02.json`, and prints an `[ATTENTION]` line saying so.

Sections 2 and 3 send 19 requests between them. Section 4 sends nothing at all: it trains a model on the machine this notebook is running on.

In [1]:
import hashlib
import json
import os
import re
import sqlite3
import urllib.error
import urllib.request
from contextlib import closing
from datetime import date
from enum import Enum
from pathlib import Path
from typing import Literal, Optional, get_args

import openai
import pandas as pd
import tiktoken
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# The model this seminar is pinned to - the same one notebook 01 used.
MODEL = "gpt-5.4-nano-2026-03-17"

# List prices as of 15 September 2026, in US dollars per million tokens.
PRICE_IN_USD_PER_MTOK = 0.20       # what you pay for what you send
PRICE_CACHED_USD_PER_MTOK = 0.02   # ... for the part of it the provider has cached from an earlier request
PRICE_OUT_USD_PER_MTOK = 1.25      # what you pay for what comes back

DATA_DIR = Path("data")      # the notebook's data files, next to it
LOCAL_DIR = Path("_local")   # where they are downloaded to when data/ is not there, as in Colab; never committed
RAW_DATA_URL = ("https://raw.githubusercontent.com/simonhatzesberger/advanced-genai-actuarial/main/"
                "notebooks/02_llm_techniques/data/")
RECORDINGS_SHA256 = "c70351c06e860dda96717a0677e30f657d3fc5c22398f24fcd401a6b85d74248"  # data/recordings_02.json: the replies of the saved run


def find_api_key(name: str = "OPENAI_API_KEY") -> str | None:
    """Return the key if this environment can see one, otherwise None. Never raises."""
    try:
        from google.colab import userdata  # available on Colab only

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        # Not on Colab, or the secret does not exist, or notebook access is off.
        pass
    return os.environ.get(name) or None


key = find_api_key()
client = OpenAI(api_key=key) if key else None


def sha256_of(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def data_file(name: str, expected_sha256: str | None = None) -> Path | None:
    """A data file of this notebook: from data/ next to it, otherwise downloaded once from GitHub into _local/data/.
    The download is written to a temporary name and renamed only once it is complete."""
    for candidate in (DATA_DIR / name, LOCAL_DIR / "data" / name):
        if candidate.exists():
            path = candidate
            break
    else:
        path = LOCAL_DIR / "data" / name
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(name + ".part")
        try:
            with urllib.request.urlopen(RAW_DATA_URL + name, timeout=60) as response:
                partial.write_bytes(response.read())
            os.replace(partial, path)  # atomic: the file appears under its name only once it is complete
        except Exception as exc:
            partial.unlink(missing_ok=True)
            reason = f"HTTP {exc.code}" if isinstance(exc, urllib.error.HTTPError) else type(exc).__name__
            print(f"[ATTENTION] data/{name} is not next to this notebook, and the download from GitHub failed ({reason}).")
            print(f"            Download it by hand from {RAW_DATA_URL + name}")
            print("            and put it into a folder called data/ next to this notebook (in Colab: the folder icon on the")
            print("            left), then run the cell again.")
            return None
    if expected_sha256 is not None and sha256_of(path) != expected_sha256:
        print(f"[ATTENTION] {path.as_posix()} is not byte-identical to the published file (its checksum differs).")
        print("            Delete it and run the cell again to download a fresh copy.")
        return None
    return path


def short_error(exc: Exception) -> str:
    """One line describing a failed call. It never repeats a key, not even a masked one."""
    text = re.sub(r"sk-[A-Za-z0-9_\-*]{4,}", "sk-***", " ".join(str(exc).split()))
    return f"{type(exc).__name__}: {text[:160]}"


LEDGER: list[dict] = []  # one row per call this session paid for


def log_usage(purpose: str, usage: dict) -> dict:
    """Record one paid call in the ledger and return its row, with its cost in USD at the list prices above."""
    cost = ((usage["input_tokens"] - usage["cached_tokens"]) * PRICE_IN_USD_PER_MTOK
            + usage["cached_tokens"] * PRICE_CACHED_USD_PER_MTOK
            + usage["output_tokens"] * PRICE_OUT_USD_PER_MTOK) / 1e6
    no_cache = (usage["input_tokens"] * PRICE_IN_USD_PER_MTOK + usage["output_tokens"] * PRICE_OUT_USD_PER_MTOK) / 1e6
    row = {"purpose": purpose, **usage, "cost_usd": cost, "cost_usd_no_cache": no_cache}
    LEDGER.append(row)
    return row


def spend_report() -> pd.DataFrame:
    """Calls, tokens and cost of everything this session has paid for, by purpose."""
    counts = ["calls", "input_tokens", "cached_tokens", "output_tokens"]
    table = pd.DataFrame(LEDGER).groupby("purpose", sort=False).agg(
        calls=("cost_usd", "size"), input_tokens=("input_tokens", "sum"), cached_tokens=("cached_tokens", "sum"),
        output_tokens=("output_tokens", "sum"), cost_usd=("cost_usd", "sum"),
        cost_usd_no_cache=("cost_usd_no_cache", "sum"))
    table.loc["total"] = table.sum()
    table[counts] = table[counts].astype(int)
    return table.round({"cost_usd": 5, "cost_usd_no_cache": 5})


# Every request goes through respond(). With a key it calls the model; without one - or when the call fails - it
# replays the reply that the saved run got to the identical request, from data/recordings_02.json, and says so.
SESSION_REPLIES: dict[str, dict] = {}  # this session's live replies, by request fingerprint
_RECORDED: dict = {}                   # the saved run's replies, read on first use
_ANNOUNCED: set = set()


def fingerprint(request: dict) -> str:
    """A checksum of everything that decides a reply: model, instructions, input, tools and schema."""
    text = json.dumps({"model": MODEL, **request}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def recorded_reply(fp: str, purpose: str, reason: str) -> dict:
    """The saved run's reply to the same request, announced by an [ATTENTION] line (once per cell and purpose)."""
    if not _RECORDED:
        path = data_file("recordings_02.json", expected_sha256=RECORDINGS_SHA256)
        if path is None:
            raise FileNotFoundError("recordings_02.json is needed without a key - see the message above.")
        _RECORDED.update(json.loads(path.read_text(encoding="utf-8")))
    if fp not in _RECORDED["replies"]:
        raise RuntimeError(f"{reason}, and the saved run has no reply to this exact request: a changed prompt needs "
                           "a key - see notebook 00, section 6.")
    try:
        cell = get_ipython().execution_count
    except NameError:
        cell = 0
    if (cell, purpose) not in _ANNOUNCED:
        _ANNOUNCED.add((cell, purpose))
        print(f"[ATTENTION] {reason}: '{purpose}' shows the reply the saved run got on "
              f"{_RECORDED['meta']['recorded_on']} (data/recordings_02.json).")
    return _RECORDED["replies"][fp]


def as_input_item(item) -> dict:
    """An output item of a reply, in the form in which it goes back to the model in the next request."""
    if item.type == "function_call":
        return {"type": "function_call", "call_id": item.call_id, "name": item.name, "arguments": item.arguments}
    if item.type == "message":
        return {"role": "assistant", "content": "".join(p.text for p in item.content if p.type == "output_text")}
    return item.model_dump(exclude_none=True)  # anything else goes back as it came


def respond(purpose: str, schema: type[BaseModel] | None = None, **request) -> dict:
    """Send one request to the model and return the reply as a plain dict: its text, its output items, the function
    calls among them and its token counts. A `schema` goes to client.responses.parse as text_format."""
    fp = fingerprint({**request, "schema": schema.model_json_schema() if schema else None})
    if client is None:
        return recorded_reply(fp, purpose, "No key")
    try:
        if schema is None:
            response = client.responses.create(model=MODEL, **request)
        else:
            response = client.responses.parse(model=MODEL, text_format=schema, **request)
    except openai.APIError as exc:  # no credit left, the API not answering, ...
        return recorded_reply(fp, purpose, f"The call failed ({short_error(exc)})")
    output = [as_input_item(item) for item in response.output]
    details = response.usage.input_tokens_details
    reply = {
        "purpose": purpose,
        "output_text": response.output_text or "",
        "output": output,
        "calls": [item for item in output if item.get("type") == "function_call"],
        "usage": {"input_tokens": response.usage.input_tokens,
                  "cached_tokens": (details.cached_tokens or 0) if details else 0,
                  "output_tokens": response.usage.output_tokens},
    }
    log_usage(purpose, reply["usage"])
    SESSION_REPLIES[fp] = reply  # kept, so that a live run can be recorded
    return reply


def ask(prompt, instructions=None, purpose="ask"):
    """Send one prompt to the model and return its answer as plain text."""
    options = {} if instructions is None else {"instructions": instructions}
    return respond(purpose, input=prompt, **options)["output_text"].strip()


def parse(prompt, schema, instructions=None, purpose="parse"):
    """Send one prompt with a Pydantic class as text_format and return the validated object."""
    options = {} if instructions is None else {"instructions": instructions}
    reply = respond(purpose, schema=schema, input=prompt, **options)
    return schema.model_validate_json(reply["output_text"])  # the object client.responses.parse calls output_parsed


if key:
    print(f"[PASS] A key named OPENAI_API_KEY is visible ({len(key)} characters long).")
    print("       Its value is never printed - not here, not anywhere.")
else:
    print("[ATTENTION] No key named OPENAI_API_KEY is visible to this notebook.")
    print("            Sections 2 and 3 show the replies of the saved run instead, from data/recordings_02.json;")
    print("            notebook 00, section 6 explains how to add a key.")

print(f"Model  : {MODEL}")
print(f"Prices : USD {PRICE_IN_USD_PER_MTOK} in (USD {PRICE_CACHED_USD_PER_MTOK} if cached) / "
      f"USD {PRICE_OUT_USD_PER_MTOK} out per 1M tokens")
print("         (list price as of 15 September 2026)")
print("Note   : section 4 needs no key at all - it trains a model on this machine.")

[PASS] A key named OPENAI_API_KEY is visible (164 characters long).
       Its value is never printed - not here, not anywhere.
Model  : gpt-5.4-nano-2026-03-17
Prices : USD 0.2 in (USD 0.02 if cached) / USD 1.25 out per 1M tokens
         (list price as of 15 September 2026)
Note   : section 4 needs no key at all - it trains a model on this machine.


---

## 2 Structured outputs, in three stages

The prompt-engineering section of `01_llm_basics` ended on a complaint: three prompting techniques all understood the same claim note, and not one of them returned a value you could write straight into a database column. Prompting can ask nicely; it cannot guarantee.

This section closes that gap on a longer document: a car-insurance contract. We extract from it three times over, each stage stricter than the last — free text, then a flat schema with types, then a nested schema, run over ten contracts and checked by code. Watch what changes between the stages. It is not the model's understanding. It is whether the answer is something your next line of code can use — and check.

### The contracts, and the part of them we send

The contracts come from **RISCBAC**, a corpus of 10,000 synthetic automobile insurance contracts in English and French that David Beauchemin and Richard Khoury (2023) generated on the model of Quebec's standard owner's policy form, published under CC BY 4.0. `data/` holds the first ten English ones. They describe no real person, vehicle or insurer, but they read like the real thing: declarations filled in for one customer, followed by the general conditions and notices that every contract repeats.

A whole contract runs to 157,208 to 160,536 characters. Everything we want is on its declarations pages, so we cut each contract to those pages before sending it: from the page header that names the form and the policy number, through `DECLARATIONS`, up to the privacy notice that starts at `DISCLOSURE`. Three reasons:

- **Cost.** You pay for every input token, on every call.
- **Focus.** The general conditions are full of amounts and limits that are not this contract's. Text the model never sees cannot mislead it.
- **Review.** A person can check an answer against four pages of declarations; nobody checks it against the whole contract.

The cell counts the tokens both ways with the tokeniser of notebook 01, without calling the model, and prints contract 0 as the model will see it.

In [2]:
RISCBAC_SHA256 = "a2d76ac5884f66bd65740c5c906fff4c5ac82b37bcdfd037b056a20d7c1607a7"
riscbac_path = data_file("riscbac_en_first10.jsonl", expected_sha256=RISCBAC_SHA256)
if riscbac_path is None:
    raise FileNotFoundError("riscbac_en_first10.jsonl is needed from here on - see the message above.")
contracts = [json.loads(line)["text"] for line in riscbac_path.read_text(encoding="utf-8").splitlines()]


def declarations(text: str) -> str:
    """The declarations pages of a RISCBAC contract: from the page header just before 'DECLARATIONS', which names
    the form and the policy number, up to the privacy notice that starts at 'DISCLOSURE'."""
    start = text.rindex("QUEBEC AUTOMOBILE INSURANCE POLICY", 0, text.index("DECLARATIONS"))
    return text[start:text.index("DISCLOSURE", start)]


policy_texts = [declarations(text) for text in contracts]
policy_text = policy_texts[0]  # the contract of stages 1 and 2

encoding = tiktoken.get_encoding("o200k_base")  # the tokeniser of notebook 01
whole = [len(encoding.encode(text)) for text in contracts]
cut = [len(encoding.encode(text)) for text in policy_texts]

print(f"[PASS] {riscbac_path.as_posix()}: {len(contracts)} contracts, sha256 {RISCBAC_SHA256[:8]}... as published.")
print(f"Whole contracts      : {min(whole):,} to {max(whole):,} tokens each, {sum(whole):,} for all ten")
print(f"Declarations pages   : {min(cut):,} to {max(cut):,} tokens each, {sum(cut):,} for all ten")
print(f"Sent to the model    : {sum(cut) / sum(whole):.1%} of the tokens - {sum(whole) / sum(cut):.0f} times fewer")
print(f"\nContract 0 as the model will see it, {len(policy_text):,} characters (<###NEW_PAGE###> is a page break):\n")
print(policy_text)

[PASS] data/riscbac_en_first10.jsonl: 10 contracts, sha256 a2d76ac5... as published.
Whole contracts      : 34,032 to 34,900 tokens each, 345,703 for all ten
Declarations pages   : 987 to 1,257 tokens each, 10,973 for all ten
Sent to the model    : 3.2% of the tokens - 32 times fewer

Contract 0 as the model will see it, 4,751 characters (<###NEW_PAGE###> is a page break):

QUEBEC AUTOMOBILE INSURANCE POLICY
Q.P.F. B 1 - OWNER'S FORM
Policy Number: 382787-001

DECLARATIONS
ISSUE of your new insurance policy
Item 1. NAME AND ADDRESS OF THE NAMED INSURED*
Ryan Craig
6722 Kim Hollow Suite 899
Shirleyburgh, NL B6E 2C9
*The described vehicle is and will be mainly used, stored and parked in the town/city and province shown in Item 1. If not, the client or the named insured must so declare.
Item 2. CONTRACT PERIOD
FROM: 2022-12-25* TO: 2023-12-25* EXCLUSIVELY
*at 12:01 A.M. standard time at the address of the named insured.
Item 3. PARTICULARS OF THE DESCRIBED VEHICLE 1
Acura - RDX - 2010 - N

The cut keeps 3.2% of the tokens — 10,973 instead of 345,703 for the ten contracts, 32 times fewer — and every fact this section asks for is still in it.

Read contract 0 before the model does, because it has traps for any reader. Premiums stand at the end of their lines, and so do other amounts: on the Q.E.F. N 27 line, `75000$` is a limit, and in the N 33 block, `4 47$` means four events a year and a premium of 47 dollars. The tax is printed as `94.22999999999999$`, a floating-point artefact of the generator. And the serial number is labelled in French, `Numéro de série`.

### Stage 1 — free text

First, ask for four facts: the driver, the vehicle, the premium and the contract period. The request goes into `instructions` and the contract into `input` — the same split as in `01_llm_basics`.

In [3]:
free_text = ask(
    policy_text,
    instructions=("You are an underwriter. State the driver, the vehicle, the annual premium "
                  "and the contract period."),
    purpose="stage 1: free text",
)

print(free_text)

- **Driver:** Ryan Craig  
- **Vehicle:** 2010 Acura RDX (Serial: UE4SMUNYHL50SETQ0)  
- **Annual premium:** **1047$ (excluding tax)**  
- **Contract period:** **2022-12-25 to 2023-12-25 (12:01 A.M. local time, exclusively)**


Every fact we asked for is there and correct. It is still useless to the system waiting downstream.

The premium came back as the text `**1047$ (excluding tax)**`: a number, a currency sign after it, a remark in brackets and two pairs of asterisks for bold. The contract period is one string holding two dates and a note on the time of day. The model added the serial number, which nobody asked for, and it turned the contract's "12:01 A.M. standard time" into "12:01 A.M. local time" — not the same thing while the clocks are on summer time.

You could write a parser for that. But prose carries no contract: the labels, the bold, the order and the wording are this call's choices, and the next call is free to make different ones. The fix is not a better prompt. It is to stop asking for text.

### Stage 2 — a flat schema with types

Describe the fields you want as a `pydantic` class and pass it as `text_format` to `client.responses.parse`; `parse()` from the setup cell does that. The provider constrains the model's output to the class's JSON schema, so what comes back always fits it, and `pydantic` turns it into a Python object.

Nine fields, one level deep, each with the type of its value: the dates are `date`, the premium an `int`, the tax a `float`, the number of claims an `int` that cannot be negative. The policy number is the one field that looks like a number and is not one: it is an identifier, so it stays a `str`.

In [4]:
class ContractBasics(BaseModel):
    policy_number: str = Field(description="As printed, e.g. '382787-001'; keep leading zeros")
    driver_name: str
    driver_birth_date: date
    vehicle: str = Field(description="Make, model and model year")
    period_start: date
    period_end: date
    annual_premium_cad: int = Field(description="Annual premium excluding tax, in Canadian dollars")
    tax_cad: float
    claims_last_5_years: int = Field(ge=0, description="Losses or claims reported over the past five years")


basics = parse(
    policy_text,
    ContractBasics,
    instructions="Extract these fields from the declarations of a car-insurance policy.",
    purpose="stage 2: flat schema",
)

print(basics.model_dump_json(indent=2))

{
  "policy_number": "382787-001",
  "driver_name": "Ryan Craig",
  "driver_birth_date": "1984-07-27",
  "vehicle": "Acura RDX 2010",
  "period_start": "2022-12-25",
  "period_end": "2023-12-25",
  "annual_premium_cad": 1047,
  "tax_cad": 94.23,
  "claims_last_5_years": 0
}


Every value came back with its type: `period_start` is a `date`, not a string that looks like one, and the premium an `int`. The policy number kept its form, `382787-001`. The model even tidied the tax: the contract prints `94.22999999999999$`, and `tax_cad` holds `94.23`.

A type is a promise you can compute with. The next cell works on the object as it came back — date arithmetic and a division, with no parsing at all.

In [5]:
def age_at(birth: date, on: date) -> int:
    """Age in completed years on a given date."""
    return on.year - birth.year - ((on.month, on.day) < (birth.month, birth.day))


print(f"period_start is a {type(basics.period_start).__name__}, annual_premium_cad an "
      f"{type(basics.annual_premium_cad).__name__}, tax_cad a {type(basics.tax_cad).__name__}.\n")
print(f"Driver's age at inception : {age_at(basics.driver_birth_date, basics.period_start)}")
print(f"Length of the contract    : {(basics.period_end - basics.period_start).days} days")
print(f"Tax rate                  : {basics.tax_cad / basics.annual_premium_cad:.2%}")
print(f"Premium including tax     : CAD {basics.annual_premium_cad + basics.tax_cad:,.2f}")

period_start is a date, annual_premium_cad an int, tax_cad a float.

Driver's age at inception : 38
Length of the contract    : 365 days
Tax rate                  : 9.00%
Premium including tax     : CAD 1,141.23


The driver was 38 at inception, the contract runs for 365 days, and the tax is 9.00% of the premium: three facts that no field holds, each derived by code in one line.

A flat schema has its limits, though. `vehicle` is still one string, `"Acura RDX 2010"`, in an order the model chose. And the declarations hold far more than nine values: five coverage lines, each with its own amount of insurance, deductible or premium, several endorsements, discounts. One level of fields has nowhere to put a list.

### Stage 3 — a nested schema, over all ten contracts

`CarInsuranceContract` holds what the declarations hold, with the types an actuary would give it:

- **Nested objects.** `Driver` and `Vehicle` are classes of their own inside the contract.
- **Lists of nested objects.** `coverages` holds one `CoverageLine` for Section A and one for each of Protections 1 to 4 of Section B; `endorsements` holds one `Endorsement` per Q.E.F. endorsement, and `discounts` one `Discount` each. How many entries a list has is the contract's business, not the schema's.
- **`Optional` fields** say "may be absent" instead of forcing an invented value: a deductible on a protection that is excluded, a creditor where the vehicle is not financed. In the JSON schema they become "a number or `null`" — required, but allowed to be empty.
- **Booleans** — `quebec_form`, `included`, `licence_suspended_last_3_years` — for facts that are yes or no.
- **Fixed categories.** The province and the section are `Literal`s; how the vehicle is held is an `Enum` that maps the contract's wording ("Purchase used", "Rental used") onto codes of your own. Both become an `enum` in the JSON schema, and the model can return only one of the listed values. The endorsement code is a `str` with a `pattern`: it must be digits and at most one letter, such as `27` or `23a`, never `N 27`.
- **An ordered level.** `physical_damage_cover` runs from `none` through `specified_perils` and `all_perils_except_collision` to `all_risks`. A `Literal` knows no order, so `RANK` gives it one.
- **A home for every number.** `Endorsement` has a `limit_cad` as well as a `premium_cad`, so that the 75,000 of Q.E.F. 27 has somewhere to go that is not a premium.
- **Dates and numbers**, as in stage 2.

The descriptions in `Field(...)` are part of the prompt: the model reads them together with the schema. That is where the rule for the level is written down, and the warning that a limit is never a premium.

In [6]:
class Acquisition(str, Enum):  # an Enum: the contract's wording, mapped onto codes of your own
    PURCHASED_NEW = "purchased_new"
    PURCHASED_USED = "purchased_used"
    LEASED_NEW = "leased_new"
    LEASED_USED = "leased_used"


Province = Literal["AB", "BC", "MB", "NB", "NL", "NS", "NT", "NU", "ON", "PE", "QC", "SK", "YT"]
CoverLevel = Literal["none", "specified_perils", "all_perils_except_collision", "all_risks"]  # ordered, low to high
RANK = {level: rank for rank, level in enumerate(get_args(CoverLevel))}


class Driver(BaseModel):
    name: str
    birth_date: date
    licence_suspended_last_3_years: bool
    claims_last_5_years: int = Field(ge=0)


class Vehicle(BaseModel):
    make: str
    model: str
    model_year: int
    serial_number: str
    acquisition: Acquisition = Field(description="The contract's 'Purchase new', 'Purchase used', 'Rental new' or "
                                                 "'Rental used', as purchased_new, purchased_used, leased_new or "
                                                 "leased_used")
    interest_holder: Optional[str] = Field(description="Name of the creditor or lessor entitled to Chapter B "
                                                       "benefits; null if there is none")


class CoverageLine(BaseModel):
    section: Literal["A", "B1", "B2", "B3", "B4"] = Field(description="Section A, or Protection 1 to 4 of Section B")
    included: bool = Field(description="False only if the line says 'Excluded'")
    amount_of_insurance_cad: Optional[int] = Field(description="Amount of insurance; null if none is shown")
    deductible_cad: Optional[int] = Field(description="Deductible per claim; null if none is shown")
    premium_cad: Optional[int] = Field(description="Premium charged for this line; null if the line says "
                                                   "'Included' or 'Excluded' or shows no premium")


class Endorsement(BaseModel):
    code: str = Field(pattern=r"^[0-9]+[a-z]?$", description="The Q.E.F. number alone, e.g. '34' or '23a'")
    title: str
    limit_cad: Optional[int] = Field(description="The amount of insurance or limit printed for the endorsement; "
                                                 "null if none is shown")
    premium_cad: Optional[int] = Field(description="Premium charged for the endorsement; null if it says 'Included' "
                                                   "or shows no premium. A limit, an amount of insurance, an amount "
                                                   "per day or a number of events is never a premium.")


class Discount(BaseModel):
    reason: str
    percent: int


class CarInsuranceContract(BaseModel):
    policy_number: str = Field(description="As printed, e.g. '382787-001'; keep leading zeros")
    quebec_form: bool = Field(description="True if the policy is written on the Quebec owner's form (Q.P.F.)")
    province: Province = Field(description="Province of the named insured's address in Item 1")
    period_start: date
    period_end: date
    driver: Driver = Field(description="The main driver in Item 6")
    vehicle: Vehicle
    coverages: list[CoverageLine]
    physical_damage_cover: CoverLevel = Field(description="The widest Section B protection included in the coverage "
                                              "lines: all_risks if B1 is included, else all_perils_except_collision "
                                              "if B3 is, else specified_perils if B4 is, else none")
    endorsements: list[Endorsement]
    discounts: list[Discount]
    annual_premium_cad: int = Field(description="Annual premium excluding tax")
    tax_cad: float


CONTRACT_INSTRUCTIONS = (
    "You read the declarations of a Quebec automobile insurance policy and fill in the schema. Copy names, numbers "
    "and identifiers exactly as printed. Amounts are in Canadian dollars. Give one coverage line for Section A and "
    "one for each of Protections 1 to 4 of Section B, and one endorsement for every Q.E.F. endorsement. Use null "
    "where the declarations show no value."
)

schema = CarInsuranceContract.model_json_schema()
print(f"CarInsuranceContract: {len(schema['properties'])} fields, {len(schema['$defs'])} nested definitions, "
      f"{len(json.dumps(schema)):,} characters of JSON schema.\n")
print("province    ->", schema["properties"]["province"]["enum"])  # a Literal becomes an enum in the JSON schema ...
print("acquisition ->", schema["$defs"]["Acquisition"]["enum"])    # ... and so does an Enum
print("cover level ->", RANK)

CarInsuranceContract: 13 fields, 6 nested definitions, 5,227 characters of JSON schema.

province    -> ['AB', 'BC', 'MB', 'NB', 'NL', 'NS', 'NT', 'NU', 'ON', 'PE', 'QC', 'SK', 'YT']
acquisition -> ['purchased_new', 'purchased_used', 'leased_new', 'leased_used']
cover level -> {'none': 0, 'specified_perils': 1, 'all_perils_except_collision': 2, 'all_risks': 3}


Ten contracts, ten calls, with the same schema and instructions each time. The cell prints one line per contract, with the level the model gave it, and then contract 3 in full: the one whose Protections 1 and 2 are excluded.

In [7]:
extracted: dict[int, CarInsuranceContract] = {}
for number, text in enumerate(policy_texts):
    extracted[number] = parse(text, CarInsuranceContract, instructions=CONTRACT_INSTRUCTIONS,
                              purpose="stage 3: nested schema")
    c = extracted[number]
    vehicle = f"{c.vehicle.make} {c.vehicle.model} {c.vehicle.model_year}"
    print(f"Contract {number}: {c.policy_number}  {c.province}  {c.driver.name:<19} {vehicle:<43} "
          f"{c.physical_damage_cover:<28} CAD {c.annual_premium_cad:>5,}")

print(f"\n[PASS] {len(extracted)} of {len(policy_texts)} replies are valid {CarInsuranceContract.__name__} objects.")
print("\nContract 3 in full:\n")
print(extracted[3].model_dump_json(indent=1))

Contract 0: 382787-001  NL  Ryan Craig          Acura RDX 2010                              all_perils_except_collision  CAD 1,047
Contract 1: 476236-001  BC  Kevin Parrish       Genesis G90 2017                            all_risks                    CAD 1,035
Contract 2: 518406-001  NB  Janice Garner       Subaru Outback 2001                         all_risks                    CAD   911
Contract 3: 472231-001  SK  Bryan Clark         Mercedes-Benz M-Class 2003                  specified_perils             CAD   721
Contract 4: 106344-001  ON  Elizabeth Williams  Mercedes-Benz GLS 2020                      all_risks                    CAD   901
Contract 5: 221325-001  MB  Felicia Weber       Chevrolet Silverado 2500 Extended Cab 2003  all_risks                    CAD   952
Contract 6: 367030-001  NB  William Green       Oldsmobile 98 1995                          all_perils_except_collision  CAD 1,061
Contract 7: 498106-001  AB  Patrick Munoz       BMW X6 2017                        

All ten replies are valid `CarInsuranceContract` objects: every date a date, every province one of the thirteen, every endorsement code of the form the pattern demands. That is the promise of structured outputs, and it held ten times out of ten.

Now read contract 3. Its lines are right: `included` is false for B1 and B2, with nulls where the contract shows nothing; B3 has its deductible of 250 and its premium of 218; the 75,000 of Q.E.F. 27 went into `limit_cad`, not into the premium. But its level is `specified_perils`, although B3, all perils except collision, is included — by the model's own lines, the level is `all_perils_except_collision`. The list above shows three more such labels: contracts 0, 6 and 7 are marked `all_perils_except_collision`, although Protection 1, all risks, is included in all three. Nothing in the schema can see that.

### What code can check

A schema guarantees the *shape* of a value. It never guarantees that the value is right. But once the values have types, code can test them — against arithmetic that must hold, against each other and against the text they came from. None of the ten checks below needs an answer key:

- **Arithmetic.** The premiums of the coverage lines and endorsements must add up to the annual premium. The tax must be 9.0% of it, the rate that every contract of the sample uses. The contract must run for 365 days.
- **Consistency.** The policy number, the serial number and the birth date must appear in the contract character for character: the model copied them rather than made them up. The policy number must have the form of one. The cover level must follow from the coverage lines by the rule in its description. The acquisition code must match the contract's wording. And an interest holder must go with Q.E.F. 5a (a leased vehicle) or 23a (a financed one), and only with those.
- **Plausibility.** The main driver should be at least 16 at inception, and a policy on the Quebec form should insure someone whose address is in Quebec.

In [8]:
checks = []  # one row per check, for the record


def check(label: str, failed: list[int]) -> None:
    """Print one check over all contracts: [PASS] if none failed, otherwise [ATTENTION] naming the contracts."""
    n = len(extracted)
    checks.append({"check": label, "passed": n - len(failed), "of": n, "attention": failed})
    if failed:
        print(f"[ATTENTION] {label}: {n - len(failed)} of {n} - contracts {', '.join(map(str, failed))}")
    else:
        print(f"[PASS]      {label}: {n} of {n}")


def line_premiums(c: CarInsuranceContract) -> int:
    return sum(line.premium_cad or 0 for line in c.coverages) + sum(e.premium_cad or 0 for e in c.endorsements)


def implied_level(c: CarInsuranceContract) -> str:
    """The cover level that the coverage lines imply, by the rule in the schema's description."""
    included = {line.section for line in c.coverages if line.included}
    for section, level in [("B1", "all_risks"), ("B3", "all_perils_except_collision"), ("B4", "specified_perils")]:
        if section in included:
            return level
    return "none"


WORDING = {Acquisition.PURCHASED_NEW: "Purchase new", Acquisition.PURCHASED_USED: "Purchase used",
           Acquisition.LEASED_NEW: "Rental new", Acquisition.LEASED_USED: "Rental used"}


def grounded(c: CarInsuranceContract, text: str) -> bool:
    """Do the identifiers the model returned appear, character for character, in the contract it read?"""
    return all(value in text for value in (c.policy_number, c.vehicle.serial_number, c.driver.birth_date.isoformat()))


E = list(extracted.items())
print("Arithmetic")
check("line premiums add up to the annual premium",
      [n for n, c in E if line_premiums(c) != c.annual_premium_cad])
check("tax is 9.0% of the annual premium, to the cent",
      [n for n, c in E if abs(c.tax_cad - 0.09 * c.annual_premium_cad) >= 0.005])
check("the contract runs for 365 days",
      [n for n, c in E if (c.period_end - c.period_start).days != 365])
print("Consistency")
check("policy number, serial number and birth date appear in the contract",
      [n for n, c in E if not grounded(c, policy_texts[n])])
check("policy number has the form 123456-001",
      [n for n, c in E if not re.fullmatch(r"\d{6}-\d{3}", c.policy_number)])
check("cover level agrees with the coverage lines",
      [n for n, c in E if implied_level(c) != c.physical_damage_cover])
check("acquisition matches the wording in Item 3",
      [n for n, c in E if WORDING[c.vehicle.acquisition] not in policy_texts[n]])
check("an interest holder exactly when Q.E.F. 5a or 23a is attached",
      [n for n, c in E if (c.vehicle.interest_holder is not None) != bool({e.code for e in c.endorsements} & {"5a", "23a"})])
print("Plausibility")
check("main driver at least 16 years old at inception",
      [n for n, c in E if age_at(c.driver.birth_date, c.period_start) < 16])
check("an address in Quebec for a policy on the Quebec form",
      [n for n, c in E if c.quebec_form and c.province != "QC"])

number = extracted[9].policy_number
print(f"\nContract 9's policy number is {number!r}. Stored as a number, its first part would be "
      f"{int(number.split('-')[0])}: the leading zero would be gone.")

Arithmetic
[ATTENTION] line premiums add up to the annual premium: 4 of 10 - contracts 1, 2, 6, 7, 8, 9
[PASS]      tax is 9.0% of the annual premium, to the cent: 10 of 10
[PASS]      the contract runs for 365 days: 10 of 10
Consistency
[PASS]      policy number, serial number and birth date appear in the contract: 10 of 10
[PASS]      policy number has the form 123456-001: 10 of 10
[ATTENTION] cover level agrees with the coverage lines: 6 of 10 - contracts 0, 3, 6, 7
[PASS]      acquisition matches the wording in Item 3: 10 of 10
[PASS]      an interest holder exactly when Q.E.F. 5a or 23a is attached: 10 of 10
Plausibility
[ATTENTION] main driver at least 16 years old at inception: 9 of 10 - contracts 1
[ATTENTION] an address in Quebec for a policy on the Quebec form: 0 of 10 - contracts 0, 1, 2, 3, 4, 5, 6, 7, 8, 9

Contract 9's policy number is '040324-001'. Stored as a number, its first part would be 40324: the leading zero would be gone.


**The schema held; the arithmetic did not.** The premium check failed for six contracts and the level check for four. We compared the ten replies with the contracts field by field, outside this notebook, which prints only contract 3 in full. Every name, identifier, date, province, annual premium and tax is right, and so is every acquisition code. An interest holder is present exactly where the contract names one, but the model copied the bank's street address along with its name, and in contract 0 the whole address. The mistakes sit where the model had to decide which amount belongs to which line:

- In contracts 2, 6, 7, 8 and 9, the endorsement Q.E.F. 27 was given a premium of 750. That is the limit `Other covered expenses during a trip - 750$` of the travel endorsement 20a, printed just above the 27 line (in contract 8, with the 23a line in between).
- In contracts 1, 6, 7 and 9, Protection 1, printed as `All risks Included`, was given the premium of Protection 2.
- In contracts 2 and 7, the premium of Q.E.F. 34, accident benefits, is missing; in contract 6 it went into `limit_cad`.
- In contract 7, the Section A premium of 500 became a deductible. In contract 6, the deductibles and premiums of Protections 2 and 3 moved up one line, and Protection 3 was marked excluded.
- In contracts 0, 3, 6 and 7, the level contradicts the model's own coverage lines.

Every contract with one of these mistakes failed at least one check, but not every mistake would have been caught on its own. Contract 6's shifted deductibles and premiums, with Protection 3 marked excluded, leave both the sum of the premiums and the level implied by the lines unchanged; the contract was flagged for its other mistakes. And no check reads `limit_cad`: for the same printed Q.E.F. 33 limits, the model returned null in contracts 0, 3, 4 and 8, 60 in contract 1 and 400 in contract 5. So contracts 4 and 5, which failed no check on account of the model, are right in every premium, deductible and coverage flag, but they do not agree on their limits. Checks correct nothing, and they see only what they test: here they turned ten replies into eight for a person to read, and named the check each one failed.

The last two `[ATTENTION]` lines are not the model's doing: **the checks found the data, not the model.** Contract 1's main driver, born on 6 October 2006, was 15 when his policy on a 2017 Genesis G90 began; and every contract is written on the Quebec form while none of the addresses is in Quebec. Both are artefacts of the synthetic generator, and the model reported them faithfully. The Quebec check could count them only because the province is an enum; ten free-text addresses would have hidden the pattern.

Two design choices made the checks possible. Every number has its own field: `limit_cad` took the 75,000 of Q.E.F. 27 in every contract that has it, so the errors that remain are errors of assignment, not of a missing field. And the policy number is a `str`: contract 9's `040324-001` kept its leading zero.

### From ten contracts to a portfolio table

The point of all this is the last step. Validated objects drop straight into `pandas` with no parsing and no cleaning, which means they drop straight into whatever your pricing or portfolio analysis reads next. `RANK` sorts the table by cover level, and `checks_flagged` counts the checks each contract failed.

In [9]:
flagged = {n: sum(n in row["attention"] for row in checks) for n in extracted}  # checks each contract failed
rows = []
for n, c in extracted.items():
    lines = {line.section: line for line in c.coverages}
    rows.append({
        "contract": n,
        "policy": c.policy_number,
        "prov": c.province,
        "driver_age": age_at(c.driver.birth_date, c.period_start),
        "vehicle": f"{c.vehicle.make} {c.vehicle.model} {c.vehicle.model_year}",
        "acquisition": c.vehicle.acquisition.value,
        "cover": implied_level(c),  # derived by code from the coverage lines, not the model's own label
        "liability_cad": lines["A"].amount_of_insurance_cad if "A" in lines else None,
        "claims_5y": c.driver.claims_last_5_years,
        "endorsements": " ".join(e.code for e in c.endorsements),
        "discount_pct": sum(d.percent for d in c.discounts),
        "premium_cad": c.annual_premium_cad,
        "checks_flagged": flagged[n],
    })
portfolio = pd.DataFrame(rows).set_index("contract").sort_values("cover", key=lambda s: s.map(RANK), kind="stable")
display(portfolio)

codes = {n: {e.code for e in c.endorsements} for n, c in extracted.items()}
has = lambda code: sum(code in codes[n] for n in codes)  # noqa: E731
print("Cover level (from the coverage lines), low to high:")
for level, group in portfolio.groupby("cover", sort=False):
    print(f"  {level:<28} {len(group):>2} contract{'s' if len(group) > 1 else ' '}, "
          f"mean premium CAD {group['premium_cad'].mean():,.0f}")
holders = sum(c.vehicle.interest_holder is not None for c in extracted.values())
print(f"Interest holder (creditor or lessor)   : {holders} of 10")
print(f"Roadside assistance (Q.E.F. 33)        : {has('33')} of 10")
print(f"Travel expenses (Q.E.F. 20a)           : {has('20a')} of 10")
print(f"Accident benefits (Q.E.F. 34)          : {has('34')} of 10")
print(f"Association discount on top of the 2%  : {(portfolio['discount_pct'] > 2).sum()} of 10")
print(f"A claim in the past five years         : {(portfolio['claims_5y'] > 0).sum()} of 10")
print(f"Main driver aged 75 or over            : {(portfolio['driver_age'] >= 75).sum()} of 10")
low, high = portfolio["premium_cad"].idxmin(), portfolio["premium_cad"].idxmax()
print(f"Premiums                               : CAD {portfolio.loc[low, 'premium_cad']:,} (contract {low}) to "
      f"CAD {portfolio.loc[high, 'premium_cad']:,} (contract {high})")

,policy,prov,driver_age,vehicle,acquisition,cover,liability_cad,claims_5y,endorsements,discount_pct,premium_cad,checks_flagged
contract,,,,,,,,,,,,
3,472231-001,SK,60,Mercedes-Benz M-Class 2003,purchased_used,all_perils_except_collision,1000000,1,27 33 34 41,2,721,2
0,382787-001,NL,38,Acura RDX 2010,purchased_used,all_risks,1000000,0,23a 27 33 34 41,2,1047,2
1,476236-001,BC,15,Genesis G90 2017,purchased_used,all_risks,1000000,0,33 41,2,1035,3
2,518406-001,NB,82,Subaru Outback 2001,purchased_used,all_risks,1000000,0,20a 27 34 41,2,911,2
4,106344-001,ON,66,Mercedes-Benz GLS 2020,purchased_used,all_risks,1000000,0,27 33 34 41,2,901,1
5,221325-001,MB,57,Chevrolet Silverado 2500 Extended Cab 2003,leased_used,all_risks,1000000,0,5a 20a 27 33 34 41,2,952,1
6,367030-001,NB,81,Oldsmobile 98 1995,purchased_used,all_risks,1000000,0,20a 27 34 41,12,1061,3
7,498106-001,AB,75,BMW X6 2017,purchased_used,all_risks,2000000,1,20a 27 34 41,2,1204,3
8,290885-001,PE,51,Ford Club Wagon 1998,purchased_used,all_risks,1000000,0,20a 23a 27 33 34 41,2,954,2


Cover level (from the coverage lines), low to high:
  all_perils_except_collision   1 contract , mean premium CAD 721
  all_risks                     9 contracts, mean premium CAD 1,014
Interest holder (creditor or lessor)   : 3 of 10
Roadside assistance (Q.E.F. 33)        : 6 of 10
Travel expenses (Q.E.F. 20a)           : 6 of 10
Accident benefits (Q.E.F. 34)          : 9 of 10
Association discount on top of the 2%  : 2 of 10
A claim in the past five years         : 2 of 10
Main driver aged 75 or over            : 3 of 10
Premiums                               : CAD 721 (contract 3) to CAD 1,204 (contract 7)


Ten contracts in, ten rows out. The `cover` column is the level that code derives from the coverage lines, not the model's own label, which the check found wrong in four contracts: where code can derive a value from facts the model copied, let code derive it.

What the table shows, in the saved run: nine contracts cover all risks, and one, contract 3, all perils except collision — the cheapest, at CAD 721. The dearest, contract 7 at CAD 1,204, is the only one with a liability limit of 2,000,000 and one of the two with a claim in the past five years. Three vehicles have a creditor or lessor with an interest in them; six contracts add roadside assistance and six travel expenses, nine accident benefits; two carry a 10% association discount on top of the 2% for pre-authorised payment. Three main drivers are 75 or older. These are ten synthetic contracts: they illustrate the pipeline and prove nothing about pricing.

Two lessons carry over to Case Study 1. **An enum is a promise that the list is complete.** A field typed as a `Literal` or an `Enum` — an `enum` in the JSON schema — can only come back as one of its permitted values, which is what made the province check countable. So if you want codes, put your codes in a `Literal`. And where the list may not be complete, as with the endorsement codes here, keep the field a `str`, constrain its form with a pattern and check its values in code: a closed list would force an endorsement it does not know into a wrong value.

---

## 3 Function calling on a mortality database

A schema fixes the *shape* of an answer. It does nothing about a model inventing the *number* inside it — and an invented mortality rate is worse than none.

So we do not let the model know the numbers. We give it two Python functions that look them up, and our own code does the looking up. The round trip has four steps:

1. **The model asks.** Your code sends the question with a description of each function — a *tool*. The model does not answer: it replies with a function call, a function name and its arguments.
2. **Your code queries.** Your code checks the arguments and runs the query. The model never touches the database.
3. **The results go back.** Your code adds the model's call and the result to the conversation and sends it again.
4. **The model answers**, from the numbers in front of it.

The data are real: the mortality tables PKV-Sterbetafel 2022 to 2025, which the German supervisor BaFin publishes each year for private health insurance: one-year death probabilities $q_x$ by sex, here for ages 0 to 102, in a small SQLite database. They are health-insurance tables, fit for look-ups and comparisons, not for pricing a life policy. `data/README.md` gives the source.

### The database, and two functions that can only read it

A model's arguments come from text that anyone could have written, so the cell below puts five defences between the model and the data. Each one would stop a mistake on its own:

- **A read-only connection** (`mode=ro` in the SQLite URI): a write fails inside SQLite, whatever the SQL says.
- **An allow-list inside the database engine** (`set_authorizer`): the connection may read the four columns of the one table and nothing else — not the `id` column, not SQLite's own catalogue, no `PRAGMA`, no `ATTACH`.
- **Parameterised SQL.** The SQL text is fixed in our code. The arguments travel as parameters (`?`) and are never pasted into it, so there is nothing to escape by hand and nothing to inject.
- **Checks before the query.** The table must be one the database holds, the sex `MALE` or `FEMALE`, and the ages within the 0 to 102 the tables cover. Anything else comes back as an error message the model can read, not as an exception.
- **Narrow functions.** There is no "run this SQL" tool. The model chooses *which* look-up to make, never *what code* runs.

Every query is logged with its SQL and its parameters.

In [10]:
MORTALITY_SHA256 = "ce714b3cbcbddafa560a74eebbfb3ffc6616e94091e83c0d8d95795b8f0a90bf"
db_path = data_file("mortality_tables.sqlite", expected_sha256=MORTALITY_SHA256)
if db_path is None:
    raise FileNotFoundError("mortality_tables.sqlite is needed from here on - see the message above.")

READABLE = {"mortality_table": {"mortality_table", "gender", "age", "probability"}}  # allow-list: table -> columns
SQL_FUNCTIONS = {"min", "max", "count"}                                            # the functions the tools use


def authorizer(action, arg1, arg2, database, trigger) -> int:
    """SQLite asks this while it compiles each statement, once for every action the statement would take:
    only SELECT, only the allowed columns, three functions."""
    if action == sqlite3.SQLITE_SELECT:
        return sqlite3.SQLITE_OK
    if action == sqlite3.SQLITE_READ and arg2 in READABLE.get(arg1, set()):
        return sqlite3.SQLITE_OK
    if action == sqlite3.SQLITE_FUNCTION and arg2.lower() in SQL_FUNCTIONS:
        return sqlite3.SQLITE_OK
    return sqlite3.SQLITE_DENY


def connect_read_only() -> sqlite3.Connection:
    connection = sqlite3.connect(f"{db_path.resolve().as_uri()}?mode=ro", uri=True)  # read-only: no write to this file can succeed
    connection.set_authorizer(authorizer)
    return connection


SQL_LOG: list[dict] = []  # every query that ran: its SQL, its parameters and how many rows came back


def query(sql: str, parameters: tuple = ()) -> list[tuple]:
    with closing(connect_read_only()) as connection:  # `with sqlite3.connect(...)` alone would not close it
        rows = connection.execute(sql, parameters).fetchall()
    SQL_LOG.append({"sql": sql, "parameters": parameters, "rows": len(rows)})
    return rows


def list_mortality_tables() -> dict:
    """The tables in the database, with the sexes and the age range of each."""
    rows = query("SELECT mortality_table, gender, MIN(age), MAX(age), COUNT(*) FROM mortality_table "
                 "GROUP BY mortality_table, gender ORDER BY mortality_table, gender")
    return {"tables": [{"table": t, "sex": s, "min_age": lo, "max_age": hi, "rows": n} for t, s, lo, hi, n in rows]}


CATALOGUE = list_mortality_tables()["tables"]
TABLES = sorted({row["table"] for row in CATALOGUE})
AGE_MIN, AGE_MAX = min(row["min_age"] for row in CATALOGUE), max(row["max_age"] for row in CATALOGUE)


def fetch_probabilities(mortality_table: str, sex: str, start_age: int, end_age: int) -> dict:
    """One-year death probabilities q_x for one table, one sex and an age range - or an error the model can read."""
    if mortality_table not in TABLES:
        return {"error": f"unknown table {mortality_table!r}", "available_tables": TABLES}
    if sex not in ("MALE", "FEMALE"):
        return {"error": f"sex must be MALE or FEMALE, not {sex!r}"}
    if not AGE_MIN <= start_age <= end_age <= AGE_MAX:
        return {"error": f"ages must satisfy {AGE_MIN} <= start_age <= end_age <= {AGE_MAX}: "
                         f"the tables end at age {AGE_MAX}"}
    rows = query("SELECT age, probability FROM mortality_table "
                 "WHERE mortality_table = ? AND gender = ? AND age BETWEEN ? AND ? ORDER BY age",
                 (mortality_table, sex, start_age, end_age))  # the arguments travel as parameters, never as SQL
    return {"table": mortality_table, "sex": sex, "rows": [{"age": age, "q_x": q} for age, q in rows]}


print(f"[PASS] {db_path.as_posix()}: sha256 {MORTALITY_SHA256[:8]}... as published.")
print(f"{len(CATALOGUE)} series ({len(TABLES)} tables x 2 sexes), ages {AGE_MIN} to {AGE_MAX}: {', '.join(TABLES)}\n")

print("fetch_probabilities('PKV-Sterbetafel 2025', 'MALE', 85, 90):")
for row in fetch_probabilities("PKV-Sterbetafel 2025", "MALE", 85, 90)["rows"]:
    print(f"  age {row['age']}: q_x = {row['q_x']:.6f}")

print("\nThe tools refuse, before any SQL runs:")
print("  ", fetch_probabilities("PKV-Sterbetafel 2025'; DROP TABLE mortality_table; --", "MALE", 85, 90)["error"])
print("  ", fetch_probabilities("PKV-Sterbetafel 2025", "FEMALE", 95, 110)["error"])

print("\nThe database refuses, whatever the SQL:")
for sql in ["DELETE FROM mortality_table", "SELECT id FROM mortality_table", "SELECT sql FROM sqlite_master"]:
    try:
        query(sql)
    except sqlite3.DatabaseError as exc:
        print(f"   {sql:<32} -> {exc}")
with closing(sqlite3.connect(f"{db_path.resolve().as_uri()}?mode=ro", uri=True)) as plain:  # no authoriser at all
    try:
        plain.execute("DELETE FROM mortality_table")
    except sqlite3.OperationalError as exc:
        print(f"   {'the same DELETE, read-only alone':<32} -> {exc}")
SQL_LOG.clear()  # the tests above are not part of the audit trail

[PASS] data/mortality_tables.sqlite: sha256 ce714b3c... as published.
8 series (4 tables x 2 sexes), ages 0 to 102: PKV-Sterbetafel 2022, PKV-Sterbetafel 2023, PKV-Sterbetafel 2024, PKV-Sterbetafel 2025

fetch_probabilities('PKV-Sterbetafel 2025', 'MALE', 85, 90):
  age 85: q_x = 0.063080
  age 86: q_x = 0.073180
  age 87: q_x = 0.084803
  age 88: q_x = 0.098028
  age 89: q_x = 0.112913
  age 90: q_x = 0.129465

The tools refuse, before any SQL runs:
   unknown table "PKV-Sterbetafel 2025'; DROP TABLE mortality_table; --"
   ages must satisfy 0 <= start_age <= end_age <= 102: the tables end at age 102

The database refuses, whatever the SQL:
   DELETE FROM mortality_table      -> not authorized
   SELECT id FROM mortality_table   -> access to mortality_table.id is prohibited
   SELECT sql FROM sqlite_master    -> access to sqlite_master.sql is prohibited
   the same DELETE, read-only alone -> attempt to write a readonly database


Each defence showed itself above: a table name carrying an SQL injection was refused as an unknown table, ages beyond 102 were refused before any SQL ran, and the engine refused a `DELETE`, the `id` column and its own catalogue — the `DELETE` twice, once by the allow-list and once, on a connection without one, by read-only mode alone.

### The two tools, as the model sees them

A tool definition is a name, a description and a JSON schema for the arguments — the same kind of schema as in section 2, now for what the model sends rather than for what it returns. `"strict": True` makes the provider hold the model to it: every argument present, each of the right type, `sex` one of two values, nothing extra. `run_tool()` runs our function for each call the model asks for, and writes the call, the SQL it led to and the number of rows into `TOOL_CALLS`: the audit trail.

In [11]:
TOOLS = [
    {
        "type": "function",
        "name": "list_mortality_tables",
        "description": "List the mortality tables in the database, with the sexes and the age range of each.",
        "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
        "strict": True,
    },
    {
        "type": "function",
        "name": "fetch_probabilities",
        "description": "One-year death probabilities q_x from the mortality database, for one table, one sex "
                       "and an age range.",
        "parameters": {
            "type": "object",
            "properties": {
                "mortality_table": {"type": "string",
                                    "description": "Exact table name, as list_mortality_tables returns it"},
                "sex": {"type": "string", "enum": ["MALE", "FEMALE"]},
                "start_age": {"type": "integer", "description": "First age, inclusive"},
                "end_age": {"type": "integer", "description": "Last age, inclusive"},
            },
            "required": ["mortality_table", "sex", "start_age", "end_age"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]
TOOL_FUNCTIONS = {"list_mortality_tables": list_mortality_tables, "fetch_probabilities": fetch_probabilities}
TOOL_INSTRUCTIONS = (
    "You answer questions about mortality tables. Every mortality rate you state must come from a tool result in "
    "this conversation; never quote a rate from memory. If a tool returns an error, say so and do not guess. "
    "Write plain markdown, without LaTeX."
)
TOOL_CALLS: list[dict] = []  # every call the model asked for, with the SQL it led to: the audit trail


def run_tool(call: dict, round_number: int = 1) -> dict:
    """Run one function call the model asked for: our function, with arguments we check - never code it wrote."""
    before = len(SQL_LOG)
    function = TOOL_FUNCTIONS.get(call["name"])
    try:
        result = function(**json.loads(call["arguments"])) if function else {"error": f"no function {call['name']!r}"}
    except (TypeError, ValueError) as exc:  # arguments that do not fit the function
        result = {"error": f"bad arguments: {exc}"}
    issued = SQL_LOG[before:]
    TOOL_CALLS.append({
        "round": round_number,
        "function": call["name"],
        "arguments": call["arguments"],
        "sql": " ".join(issued[0]["sql"].split()) if issued else None,
        "sql_parameters": issued[0]["parameters"] if issued else None,
        "rows": issued[0]["rows"] if issued else 0,
        "error": result.get("error"),
    })
    return result


for tool in TOOLS:
    arguments = ", ".join(tool["parameters"]["properties"])
    print(f"{tool['name']}({arguments})  strict={tool['strict']}")

list_mortality_tables()  strict=True
fetch_probabilities(mortality_table, sex, start_age, end_age)  strict=True


`strict` guarantees the *shape* of the arguments, not that they are safe: `mortality_table` can still be any string, which is why `fetch_probabilities` checks it against the tables that exist.

### One round trip, step by step

**Step 1 — the model asks.** Send the question with the tools. The question is a one-table version of the slide's example: men aged 85 to 100, from the 2025 table alone, as a markdown table with a short comment on the trend. The slide's comparison of 2022 with 2025 follows in the loop further below.

In [12]:
QUESTION_2025 = (
    "Fetch the one-year death probabilities for men aged 85 to 100 from the mortality table "
    "'PKV-Sterbetafel 2025'. Present them in a markdown table with the age and q_x to six decimal places, "
    "then write two sentences on the trend."
)
messages = [{"role": "user", "content": QUESTION_2025}]

first = respond("round trip: step 1", input=messages, tools=TOOLS, instructions=TOOL_INSTRUCTIONS)
print("The model came back with:", [item.get("type", "message") for item in first["output"]])

The model came back with: ['function_call']


**Still step 1 — what came back.** It did not answer. It asked for a call, and filled in the arguments itself, reading the table, the sex and the ages out of the sentence.

In [13]:
for call in first["calls"]:
    print("Function :", call["name"])
    print("Arguments:", call["arguments"])
    print("Call id  :", call["call_id"])

Function : fetch_probabilities
Arguments: {"mortality_table":"PKV-Sterbetafel 2025","sex":"MALE","start_age":85,"end_age":100}
Call id  : call_Up4pA9LPVonnMSYTCEYanZ9g


**Step 2 — your code queries.** Nothing here involves the model. It is your function, your SQL and your database — the part an auditor can re-run.

In [14]:
results = [run_tool(call) for call in first["calls"]]

for entry, result in zip(TOOL_CALLS[-len(results):], results):
    print("Function   :", entry["function"], entry["arguments"])
    if entry["error"]:
        print("Refused    :", entry["error"])
        continue
    print("SQL        :", entry["sql"])
    print("Parameters :", entry["sql_parameters"])
    print(f"Rows       : {entry['rows']}, q_x from {result['rows'][0]['q_x']} at age {result['rows'][0]['age']} "
          f"to {result['rows'][-1]['q_x']} at age {result['rows'][-1]['age']}")

Function   : fetch_probabilities {"mortality_table":"PKV-Sterbetafel 2025","sex":"MALE","start_age":85,"end_age":100}
SQL        : SELECT age, probability FROM mortality_table WHERE mortality_table = ? AND gender = ? AND age BETWEEN ? AND ? ORDER BY age
Parameters : ('PKV-Sterbetafel 2025', 'MALE', 85, 100)
Rows       : 16, q_x from 0.06308 at age 85 to 0.27302 at age 100


**Steps 3 and 4 — the results go back, and the model answers.** Append the model's own call and your result to the conversation, under the call's id, and send it again. This second request turns rows of numbers into a table and two sentences.

In [15]:
messages = messages[:1] + first["output"]  # the question, then the model's own function call(s) ...
for call, result in zip(first["calls"], results):  # ... and our result for each, under its call id
    messages.append({"type": "function_call_output", "call_id": call["call_id"], "output": json.dumps(result)})

second = respond("round trip: step 4", input=messages, tools=TOOLS, instructions=TOOL_INSTRUCTIONS)
answer_2025 = second["output_text"]
if second["calls"]:
    print("[ATTENTION] The model asked for another call instead of answering - the loop further below handles that.")
display(Markdown(answer_2025))

| Age (x) | q_x (one-year death probability) |
|---:|---:|
| 85 | 0.063080 |
| 86 | 0.073180 |
| 87 | 0.084803 |
| 88 | 0.098028 |
| 89 | 0.112913 |
| 90 | 0.129465 |
| 91 | 0.147647 |
| 92 | 0.167363 |
| 93 | 0.188107 |
| 94 | 0.209873 |
| 95 | 0.232032 |
| 96 | 0.243250 |
| 97 | 0.251290 |
| 98 | 0.258960 |
| 99 | 0.266210 |
| 100 | 0.273020 |

From age 85 to 100, the one-year death probability q_x for men rises steadily throughout the entire age range. The increase is gradual at the younger part of the interval and becomes more pronounced through the early 90s, continuing upward to age 100.

The model wrote all sixteen rates to six decimal places, from `0.063080` at age 85 to `0.273020` at 100, and two sentences on the trend. The rates came from `fetch_probabilities`, not from the model's memory. The sentences are the model's own, and thinner than the numbers: they say that the increase "becomes more pronounced through the early 90s, continuing upward to age 100", but miss that the yearly increase halves after 95, from 0.022 between 94 and 95 to 0.011 between 95 and 96. A check can confirm a number; it cannot confirm a description.

### Checking every number

Function calling makes the numbers *come from* your database. One more step shows that they *arrived* unchanged. `check_rates()` reads the model's markdown table back, row by row, and compares each $q_x$ with the database value for that age; every other decimal number in the text must be a value the tools returned, that value in per cent, or a change between two tables.

In [16]:
def series_from(results: list[dict]) -> dict[str, dict[int, float]]:
    """The q_x the tools returned in a conversation: {table: {age: q_x}}."""
    series = {}
    for result in results:
        if "rows" in result:
            series.setdefault(result["table"], {}).update({row["age"]: row["q_x"] for row in result["rows"]})
    return series


def matches(shown: str, value: float) -> bool:
    """Is `shown` (e.g. '0.063080' or '-4.3') `value` rounded to the decimals shown?"""
    decimals = len(shown.split(".")[1]) if "." in shown else 0
    return abs(float(shown) - value) <= 0.5 * 10 ** -decimals + 1e-12


def check_rates(answer: str, results: list[dict]) -> dict:
    """Compare every row of the markdown table in `answer` with the database, and every decimal in the text around
    it. A q_x must equal the database value for its age; a change in % must equal (newer / older - 1) x 100."""
    series = series_from(results)
    older, newer = (min(series), max(series)) if series else (None, None)
    values = [q for s in series.values() for q in s.values()]
    changes = [(series[newer][a] / series[older][a] - 1) * 100 for a in series.get(newer, {}) if a in series[older]]
    kinds, rows, rows_ok, text_numbers, problems = None, 0, 0, 0, []
    for line in answer.replace(chr(0x2212), "-").splitlines():  # a typographic minus sign is a minus
        if not line.strip().startswith("|"):  # text: each decimal must be a fetched q_x (or it in %) or a change
            for shown in re.findall(r"\d+\.\d+", line):
                text_numbers += 1
                if not any(matches(shown, v) for v in values + [q * 100 for q in values] + [abs(c) for c in changes]):
                    problems.append(f"text: {shown}")
            continue
        cells = [cell.strip() for cell in line.strip().strip("|").split("|")]
        if kinds is None:  # the header row says which column holds what
            kinds = ["change" if re.search(r"change|%", cell, re.IGNORECASE) else "rate" for cell in cells]
            continue
        if re.fullmatch(r":?-+:?", cells[0]):
            continue  # the separator row
        age_cell = cells[0].strip("*_ ")  # an age in bold is still an age
        if not age_cell.isdigit() or int(age_cell) not in series.get(newer, {}):
            if re.search(r"\d+\.\d+", " ".join(cells[1:])):  # a rate for an age the tools never returned
                rows += 1
                problems.append(f"row {cells[0]!r}: the tools returned no rate for this age")
            continue  # a row that states no rate, such as 'not available', has nothing to check
        age, bad = int(age_cell), []
        for kind, cell in zip(kinds[1:], cells[1:]):
            for shown in re.findall(r"-?\d+\.\d+", cell):
                if kind == "change":
                    ok = matches(shown, (series[newer][age] / series[older][age] - 1) * 100)
                else:
                    ok = any(matches(shown, s[age]) for s in series.values() if age in s)
                if not ok:
                    bad.append(shown)
        rows += 1
        rows_ok += not bad
        if bad:
            problems.append(f"age {age}: {', '.join(bad)}")
    return {"rows": rows, "rows_ok": rows_ok, "text_numbers": text_numbers, "problems": problems}


def report_rates(outcome: dict) -> None:
    if not outcome["rows"] and not outcome["text_numbers"]:
        print("[PASS]      The answer states no rate at all, so it states none that is not in the database.")
        return
    tag = "[PASS]     " if outcome["rows"] and not outcome["problems"] else "[ATTENTION]"
    text_bad = sum(problem.startswith("text:") for problem in outcome["problems"])
    print(f"{tag} Table: {outcome['rows_ok']} of {outcome['rows']} rows match the database, number for number.")
    if outcome["text_numbers"]:
        print(f"{' ' * 11} Text : {outcome['text_numbers'] - text_bad} of {outcome['text_numbers']} decimal numbers "
              "match a fetched q_x, or a change between the tables.")
    else:
        print(f"{' ' * 11} Text : no decimal numbers outside the table.")
    for problem in outcome["problems"]:
        print(f"{' ' * 12}does not match: {problem}")


rates_2025 = check_rates(answer_2025, results)
report_rates(rates_2025)

[PASS]      Table: 16 of 16 rows match the database, number for number.
            Text : no decimal numbers outside the table.


All sixteen rows match the database, number for number. The two sentences below the table quote no number, so there was nothing more to check.

### A loop, for questions that need several calls

The round trip above needed one call. A question such as "how did the rates change between the oldest and the newest table?" needs more: the model first has to find out which tables there are, then fetch two of them. `run_with_tools()` automates the four steps. It sends the conversation, runs every call the model asks for — several at once, if it asks for several — sends the results back, and repeats until the model answers. It stops after five requests whatever happens, so that a model that keeps asking cannot run up a bill. The instructions tell the model to state no rate from memory.

In [17]:
def run_with_tools(question: str, purpose: str, tools: list[dict] = TOOLS,
                   max_rounds: int = 5) -> tuple[str | None, list[dict]]:
    """Let the model call the tools until it answers, with at most `max_rounds` requests. Returns the answer and
    every tool result of the conversation."""
    conversation = [{"role": "user", "content": question}]
    results = []
    for round_number in range(1, max_rounds + 1):
        reply = respond(purpose, input=conversation, tools=tools, instructions=TOOL_INSTRUCTIONS)
        if not reply["calls"]:
            print(f"Answered after {round_number} request{'s' * (round_number > 1)} and "
                  f"{len(results)} tool call{'s' * (len(results) != 1)}.")
            return reply["output_text"], results
        conversation += reply["output"]
        for call in reply["calls"]:
            result = run_tool(call, round_number)
            results.append(result)
            conversation.append({"type": "function_call_output", "call_id": call["call_id"],
                                 "output": json.dumps(result)})
    print(f"[ATTENTION] No final answer after {max_rounds} requests: the loop stopped, as it is meant to.")
    return None, results


QUESTION_TREND = (
    "How did the one-year death probabilities of men aged 85 to 100 change between the oldest and the newest "
    "PKV-Sterbetafel in the database? Present a markdown table with the age, q_x in both tables to six decimal "
    "places and the change in per cent to one decimal place (negative if the newer table is lower), then write "
    "three sentences on the trend."
)
start = len(TOOL_CALLS)
answer_trend, results_trend = run_with_tools(QUESTION_TREND, purpose="tool loop: 2022 against 2025")
display(pd.DataFrame(TOOL_CALLS[start:]).drop(columns="sql"))
display(Markdown(answer_trend or ""))

Answered after 3 requests and 3 tool calls.


,round,function,arguments,sql_parameters,rows,error
0,1,list_mortality_tables,{},(),8,None
1,2,fetch_probabilities,"{""mortality_table"":""PKV-Sterbetafel 2022"",""sex"":""MALE"",""start_age"":85,""end_age"":100}","(PKV-Sterbetafel 2022, MALE, 85, 100)",16,None
2,2,fetch_probabilities,"{""mortality_table"":""PKV-Sterbetafel 2025"",""sex"":""MALE"",""start_age"":85,""end_age"":100}","(PKV-Sterbetafel 2025, MALE, 85, 100)",16,None


| Age | q_x (oldest: PKV-Sterbetafel 2022) | q_x (newest: PKV-Sterbetafel 2025) | Change (%) |
|---:|---:|---:|---:|
| 85 | 0.065906 | 0.063080 | -4.3 |
| 86 | 0.076309 | 0.073180 | -4.1 |
| 87 | 0.088135 | 0.084803 | -3.8 |
| 88 | 0.101452 | 0.098028 | -3.4 |
| 89 | 0.116307 | 0.112913 | -2.9 |
| 90 | 0.132707 | 0.129465 | -2.4 |
| 91 | 0.150612 | 0.147647 | -2.0 |
| 92 | 0.169904 | 0.167363 | -1.5 |
| 93 | 0.190399 | 0.188107 | -1.2 |
| 94 | 0.211354 | 0.209873 | -0.7 |
| 95 | 0.232032 | 0.232032 | 0.0 |
| 96 | 0.243250 | 0.243250 | 0.0 |
| 97 | 0.251290 | 0.251290 | 0.0 |
| 98 | 0.258960 | 0.258960 | 0.0 |
| 99 | 0.266210 | 0.266210 | 0.0 |
| 100 | 0.273020 | 0.273020 | 0.0 |

Across ages 85–94, the one-year death probabilities for men are lower in the newest table (2025) than in the oldest table (2022), with percentage decreases ranging from -4.3% to -0.7%. From age 95 onward (95–100), the values are identical in both tables, so the change is 0.0%. Overall, the downward shift fades as age increases and reaches zero by age 95.

The model needed three requests and made three tool calls. The first request asked for the list of tables; the second fetched the oldest and the newest, 2022 and 2025, in two calls at once; the third produced the answer. `TOOL_CALLS` holds each call with the round it came in, its arguments, the parameters of the SQL it led to and the number of rows. The SQL itself, left out of the display, is fixed in our code: the `GROUP BY` query of `list_mortality_tables`, and for the two `fetch_probabilities` calls the statement printed in the round trip above.

The answer: for men aged 85 to 94, $q_x$ is lower in the 2025 table than in the 2022 table, by 4.3% at 85 and by 0.7% at 94, and from 95 to 100 the two tables are identical. The tools return only rates, so the model computed those percentages itself — and they need checking as much as the rates do.

In [18]:
rates_trend = check_rates(answer_trend or "", results_trend)
report_rates(rates_trend)

[PASS]      Table: 16 of 16 rows match the database, number for number.
            Text : 3 of 3 decimal numbers match a fetched q_x, or a change between the tables.


All sixteen rows match, rates and changes alike, and so do the three decimal numbers in the text — 4.3, 0.7 and 0.0. The model's arithmetic held this time. In a trial run while this notebook was built, it wrote −3.3% for age 88, where the rates give −3.37%, and this check flagged that row and no other.

### When the database says no

The last question asks for ages the tables do not have: they end at 102. This time the model is offered only `fetch_probabilities`, so it cannot look the age range up first and step around the question.

In [19]:
QUESTION_REFUSED = (
    "What are the one-year death probabilities for women aged 98 to 105 in PKV-Sterbetafel 2025? "
    "Give them in a markdown table with q_x to six decimal places."
)
start = len(TOOL_CALLS)
answer_refused, results_refused = run_with_tools(QUESTION_REFUSED, purpose="tool loop: a request refused",
                                                 tools=[TOOLS[1]])  # fetch_probabilities only
display(pd.DataFrame(TOOL_CALLS[start:]).drop(columns="sql"))
display(Markdown(answer_refused or ""))
rates_refused = check_rates(answer_refused or "", results_refused)
report_rates(rates_refused)

Answered after 2 requests and 1 tool call.


,round,function,arguments,sql_parameters,rows,error
0,1,fetch_probabilities,"{""mortality_table"":""PKV-Sterbetafel 2025"",""sex"":""FEMALE"",""start_age"":98,""end_age"":105}",None,0,ages must satisfy 0 <= start_age <= end_age <= 102: the tables end at age 102


I can’t provide q_x for ages 98 to 105 in **PKV-Sterbetafel 2025** because the table only covers up to **age 102**. The tool returned this error:

> ages must satisfy 0 <= start_age <= end_age <= 102: the tables end at age 102

If you want, I can return the one-year death probabilities for **women aged 98 to 102** (q_98, q_99, q_100, q_101, q_102).

[PASS]      The answer states no rate at all, so it states none that is not in the database.


The model asked for women aged 98 to 105, and the function refused before any SQL ran: the table ends at 102. The model passed the error on, word for word in a quotation, stated no rate at all, and offered ages 98 to 102 instead. It did not guess the missing ages. "Do not guess" in the instructions is a request, not a guarantee, and the check below the answer is what confirms that it held.

The bill for sections 2 and 3, from the ledger of section 1:

In [20]:
if LEDGER:
    bill = spend_report()
else:
    meta = _RECORDED.get("meta", {})
    print("No paid calls in this session: every reply above was replayed from data/recordings_02.json.")
    if meta:
        no_cache = (meta["input_tokens"] * PRICE_IN_USD_PER_MTOK + meta["output_tokens"] * PRICE_OUT_USD_PER_MTOK) / 1e6
        print(f"In the saved run, these {meta['calls']} calls cost USD {meta['cost_usd']:.4f} at the list prices "
              f"of section 1, or USD {no_cache:.4f} had none of their input been cached.")
    bill = None
bill

,calls,input_tokens,cached_tokens,output_tokens,cost_usd,cost_usd_no_cache
purpose,,,,,,
stage 1: free text,1,1227,0,82,0.00035,0.00035
stage 2: flat schema,1,1419,0,90,0.00040,0.00040
stage 3: nested schema,10,23533,17920,5857,0.00880,0.01203
round trip: step 1,1,246,0,44,0.00010,0.00010
round trip: step 4,1,583,0,240,0.00042,0.00042
tool loop: 2022 against 2025,3,2222,0,603,0.00120,0.00120
tool loop: a request refused,2,490,0,160,0.00030,0.00030
total,19,29720,17920,7076,0.01156,0.01479


The saved run made 19 requests for USD 0.0116 at list prices, or USD 0.0148 had none of their input been cached. Stage 3 is three quarters of it. Each of its ten requests carried 1,180 input tokens on top of the contract and the instructions, against 10 in stage 1: the nested schema, as the provider counts it, costs about 1,170 tokens per request — more than the average contract, at 1,097. The flat schema of stage 2 cost about 200. 17,920 of stage 3's input tokens were read from the provider's cache, 1,792 per request: its instructions and schema are the same every time, and identical requests had been sent while this notebook was built. The function-calling requests of section 3 are small by comparison, and neither loop came near its cap of five requests.

**Why this matters.** The model decided *what* to look up; your code decided *how*, and did it. Every rate in the answers above came out of `fetch_probabilities` — out of a read-only database, through fixed SQL, with arguments checked first — and a few lines of code confirmed that each one arrived unchanged, and that each percentage change the model worked out from them was right. That is reproducible, testable and auditable: `TOOL_CALLS` shows every call, its arguments and the SQL it led to. The model chose the arguments, did arithmetic that code then checked, and wrote the prose around the numbers.

That division of labour is what makes function calling usable in a regulated process.

---

## 4 Fine-tuning with LoRA: a real run

Fine-tuning changes the model itself, by continuing its training on examples of the behaviour you want. Full fine-tuning updates every weight, which for anything useful means a serious GPU. **LoRA** (Low-Rank Adaptation) freezes the original weights and trains a small pair of matrices alongside them — a rounding error's worth of parameters, small enough that this next section genuinely trains, on CPU, while you watch.

The model is `Qwen/Qwen2.5-0.5B-Instruct`: 0.5 billion parameters, about 1 GB to download the first time. The task is deliberately narrow — answer a claim note with one cause-of-loss category, in a fixed format and nothing else.

No API key and no provider are involved from here on. Everything below runs locally.

In [21]:
import time
import warnings

import torch
from datasets import Dataset, disable_progress_bars
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")  # housekeeping notices only - keeps the output readable
disable_progress_bars()
torch.manual_seed(0)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

total = sum(p.numel() for p in model.parameters())
print(f"[PASS] Loaded {BASE_MODEL}: {total:,} parameters, running on CPU.")

[PASS] Loaded Qwen/Qwen2.5-0.5B-Instruct: 494,032,768 parameters, running on CPU.


### The training data

Twenty examples, written by hand for this notebook. They are **illustrative** — short invented claim notes, not anybody's claims data — and twenty is a deliberately tiny number, so that we can be honest later about what it achieves.

Each example is a prompt and the one word we want after it. Note how rigid the target is: `FIRE`, not "this looks like a fire claim".

In [22]:
EXAMPLES = [
    ("Kitchen blaze spread to the roof timbers.", "FIRE"),
    ("Burst pipe under the sink flooded the ground floor.", "WATER"),
    ("Laptop and camera taken during a break-in.", "THEFT"),
    ("Rear-ended at a red light; both bumpers dented.", "COLLISION"),
    ("Hailstones dented the bonnet and the roof.", "HAIL"),
    ("Windscreen chipped by gravel on the motorway.", "GLASS"),
    ("Electrical fault set the fuse box alight.", "FIRE"),
    ("Storm rain came through the roof into the attic.", "WATER"),
    ("Bicycle stolen from the locked garage.", "THEFT"),
    ("Side impact at a junction with a delivery van.", "COLLISION"),
    ("Hail damaged the greenhouse and the car roof.", "HAIL"),
    ("Side window smashed, nothing removed.", "GLASS"),
    ("Candle left burning scorched the living room.", "FIRE"),
    ("Washing machine hose split and soaked the floor.", "WATER"),
    ("Handbag snatched from a cafe table.", "THEFT"),
    ("Reversed into a bollard in the car park.", "COLLISION"),
    ("Hailstorm cracked the conservatory panels.", "HAIL"),
    ("Rear windscreen shattered by a falling branch.", "GLASS"),
    ("Chip pan fire damaged the kitchen units.", "FIRE"),
    ("Frozen pipe burst and flooded the cellar.", "WATER"),
]

HELD_OUT = "Tumble dryer caught fire overnight and burnt the utility room."


def build_prompt(note: str) -> str:
    return f"Claim note: {note}\nCause of loss:"


print(f"{len(EXAMPLES)} training examples. The prompt the model sees:\n")
print(build_prompt(EXAMPLES[0][0]) + f" {EXAMPLES[0][1]}")

20 training examples. The prompt the model sees:

Claim note: Kitchen blaze spread to the roof timbers.
Cause of loss: FIRE


### Before

`HELD_OUT` is a claim note the model will never be trained on. Ask the base model now, and keep the answer to compare against later.

In [23]:
def answer(note: str, max_new_tokens: int = 24) -> str:
    """Greedy generation - no sampling, so the same prompt always gives the same answer."""
    inputs = tokenizer(build_prompt(note), return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


before = answer(HELD_OUT)
print("BEFORE:", repr(before))

BEFORE: 'Fire\nDate of loss: 2019-08-31\n\nThe customer has a history of'


### The LoRA adapter

`LoraConfig` says where the adapter goes and how large it is: rank `r=8` matrices attached to the query and value projections of every attention block. `get_peft_model` freezes everything else.

`print_trainable_parameters()` is the line to look at. It reports how much of this model the training run will actually touch — and note that its "all params" is a little larger than the count printed above, because the adapter has now been added to the model.

In [24]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


### The training run

Tokenise the twenty examples, pad them to 48 tokens, and train. Everything here is kept small on purpose so it finishes on CPU: 48 tokens, batches of four, eight passes over the data.

`logging_steps=1` prints the loss after every step, so you can watch it move rather than take it on trust.

In [25]:
rows = [
    {"text": build_prompt(note) + f" {label}" + tokenizer.eos_token}
    for note, label in EXAMPLES
]

dataset = Dataset.from_list(rows).map(
    lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=48),
    batched=True,
    remove_columns=["text"],
)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="lora_cause_of_loss",
        num_train_epochs=8,
        per_device_train_batch_size=4,
        learning_rate=2e-3,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        use_cpu=True,
        seed=0,
    ),
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

started = time.time()
outcome = trainer.train()
print(f"\nTrained in {time.time() - started:.0f} seconds.")

{'loss': 6.0464, 'grad_norm': 4.703474521636963, 'learning_rate': 0.00195, 'epoch': 0.2}


{'loss': 5.3665, 'grad_norm': 4.3017048835754395, 'learning_rate': 0.0019, 'epoch': 0.4}


{'loss': 4.8447, 'grad_norm': 6.577620506286621, 'learning_rate': 0.00185, 'epoch': 0.6}


{'loss': 4.0104, 'grad_norm': 6.707264423370361, 'learning_rate': 0.0018000000000000002, 'epoch': 0.8}


{'loss': 3.0865, 'grad_norm': 6.120416641235352, 'learning_rate': 0.00175, 'epoch': 1.0}


...

{'loss': 0.3347, 'grad_norm': 2.817514419555664, 'learning_rate': 0.0002, 'epoch': 7.2}


{'loss': 0.252, 'grad_norm': 3.013723134994507, 'learning_rate': 0.00015, 'epoch': 7.4}


{'loss': 0.3635, 'grad_norm': 3.5884644985198975, 'learning_rate': 0.0001, 'epoch': 7.6}


{'loss': 0.2593, 'grad_norm': 3.5813772678375244, 'learning_rate': 5e-05, 'epoch': 7.8}


{'loss': 0.4097, 'grad_norm': 4.232916831970215, 'learning_rate': 0.0, 'epoch': 8.0}
{'train_runtime': 64.1993, 'train_samples_per_second': 2.492, 'train_steps_per_second': 0.623, 'train_loss': 1.4401521220803262, 'epoch': 8.0}

Trained in 64 seconds.


In [26]:
losses = [entry["loss"] for entry in trainer.state.log_history if "loss" in entry]

print(f"Steps        : {len(losses)}")
print(f"First loss   : {losses[0]:.4f}")
print(f"Last loss    : {losses[-1]:.4f}")
print(f"Mean loss    : {outcome.training_loss:.4f}")

Steps        : 40
First loss   : 6.0464
Last loss    : 0.4097
Mean loss    : 1.4402


A loss that falls steeply and then flattens is what a working training run looks like. It means the adapter has learnt to predict the next token of *these* examples. That is progress, but it says nothing yet about a claim note the model has never seen — which is what the next cell is for.

### After

The same held-out claim note, the same greedy generation, the same `answer()` function — the only thing that has changed is that the adapter is now trained.

In [27]:
model.eval()
after = answer(HELD_OUT)

print("Claim note:", HELD_OUT)
print()
print("BEFORE:", repr(before))
print("AFTER :", repr(after))

Claim note: Tumble dryer caught fire overnight and burnt the utility room.

BEFORE: 'Fire\nDate of loss: 2019-08-31\n\nThe customer has a history of'
AFTER : 'FIRE'


### What actually happened

Read those two lines carefully, because the honest answer is more interesting than a success story would be.

The base model was **not wrong about the category**. It said "Fire", which is correct. What it could not do was *stop*: it ran straight on into an invented date of loss and the beginning of a sentence about the customer, because nothing in its training says that "Cause of loss:" should be followed by one word and then silence.

After eight passes over twenty examples — the loss falling from about 6.0 to about 0.4 — it answers `FIRE`: the fixed vocabulary, the fixed casing, and, just as importantly, nothing else. That is a real improvement, and it is an improvement in **format**.

It is not evidence of an improvement in **accuracy**, and we should not claim one. Twenty examples cannot teach a 0.5-billion-parameter model what a cause of loss is; whatever it knew about fire and water it learnt long before we arrived. And one held-out note is a demonstration, not an evaluation — a proper one needs a held-out set of a few hundred notes and a confusion matrix.

Carry that expectation into your own work. On a small set, fine-tuning buys **consistency** reliably and **competence** rarely, which is exactly why the previous two sections matter: a schema gets you the same discipline for the price of a class definition, and no training run at all.

**Two practical notes.**

On CPU the training loop above takes a minute or two, on top of the one-off model download. In Colab, switch the runtime to a **T4 GPU** (*Runtime → Change runtime type → T4 GPU*) and the same cells finish in well under a minute — which is why nobody trains on CPU outside a teaching notebook.

And this is not the only kind of fine-tuning in the seminar. Here we trained an open-weights model ourselves, so that the mechanics are visible. Case Study 3 (`05_cs3_vehicle_damage_vision`) takes the other route: a **GPT-4o model fine-tuned on the provider's platform** for the article's research, which the seminar key calls by its model name. Same idea, no training loop, a provider's invoice instead of your own hardware.

---

## 5 Retrieval-augmented generation: a signpost

Retrieval-augmented generation (RAG) puts *your* documents in front of the model at the moment you ask: your reserving guideline, your policy wordings, last year's ORSA report. The model answers from what it has just been handed, instead of from whatever it happened to absorb during training — and it can cite which passage it used.

That makes RAG the answer to a different problem from the three above. Structured outputs and function calling fix the shape of an answer and the trustworthiness of a number; fine-tuning fixes a format or a house style. When the model is missing knowledge, none of those three help, however well you prompt — and retrieval does.

There is deliberately no code here: Case Study 2 (`04_cs2_rag_market_comparison`) builds a small RAG system end to end, over three insurers' annual reports.

---

## 6 When to use which

The four techniques are not competitors, and the commonest mistake is reaching for the expensive one. Read the middle column first: it names the problem, and the problem is what should pick the tool.

| Technique | The problem it fixes | What it costs you | When it is the wrong tool |
| --- | --- | --- | --- |
| **Structured outputs** | The answer is right but arrives as prose, so nothing downstream can read it. | A schema to write and keep in step with your data model, sent with every request as input tokens, and checks for what a schema cannot promise. | The model does not know the answer. A schema makes a wrong answer tidy, not right. |
| **Function calling** | The number must be computed, or looked up in a database you control — not generated. | Every tool is code you own, test and secure — read-only and parameterised where it touches data — and each question costs at least two requests instead of one. | The task is pure language. Wrapping a summary in a tool call buys nothing. |
| **Fine-tuning** | The model will not hold a format or a house style however you prompt it, or prompts have grown long and expensive. | Labelled examples, a training run, and a second model to version, evaluate and re-train when the base model moves on. | You need facts it never saw. Fine-tuning teaches behaviour, not knowledge — and it is the costliest way to fix a prompt. |
| **RAG** | The model is missing *your* knowledge: your wordings, your guidelines, your figures. | A document pipeline — chunking, embeddings, a store somebody has to keep current — and longer prompts. | The documents already fit in the prompt, or the problem was format rather than facts. |

A rough order of attack: prompt first, schema second, tools third, retrieval fourth, and fine-tune only when you can say which of those four failed and why.

With that, the building blocks are complete — the case studies that follow combine them.

**Next notebook:** Case Study 1, `03_cs1_claims_text_features`.